# AmazonHelp Customer Support on Twitter — Exploratory Data Analysis

**Hiver SDE Intern Take-Home Project**  
This notebook explores the AmazonHelp subset of the Kaggle *Customer Support on Twitter* dataset (thoughtvector/customer-support-on-twitter).  

### Objectives:
1. Inspect conversation threads and text distributions
2. Discover dominant intent clusters
3. Understand historical resolution patterns for retrieval-augmented generation (RAG)
4. Establish empirical justification for our 8-class intent taxonomy

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '../src')
from intent_taxonomy import INTENTS, LABELS

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

convs_path = Path('../data/conversations.csv')
df = pd.read_csv(convs_path)
print(f'Total reconstructed conversation pairs: {len(df):,}')
df.head()

## 1. Character and Word Length Distributions
Twitter's 280-character limit shapes customer interactions and agent replies.

In [ ]:
df['cust_len'] = df['customer_text'].str.len()
df['reply_len'] = df['brand_reply'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['cust_len'].clip(upper=280), bins=35, ax=axes[0], color='#1976D2')
axes[0].set_title('Customer Message Length (characters)')
axes[0].set_xlabel('Characters')

sns.histplot(df['reply_len'].clip(upper=280), bins=35, ax=axes[1], color='#388E3C')
axes[1].set_title('AmazonHelp Reply Length (characters)')
axes[1].set_xlabel('Characters')
plt.tight_layout()
plt.show()

## 2. Intent Distribution Analysis
We map customer inquiries across the 8 core operational intents defined in `src/intent_taxonomy.py`.

In [ ]:
def map_intent(text):
    text_l = str(text).lower()
    best, best_c = 'GENERAL_INQUIRY', 0
    for i in INTENTS:
        c = sum(1 for kw in i.keywords if kw in text_l)
        if c > best_c:
            best_c, best = c, i.label
    return best

df['intent'] = df['customer_text'].apply(map_intent)
counts = df['intent'].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=counts.values, y=counts.index, palette='viridis')
plt.title('Intent Distribution across AmazonHelp Conversations')
plt.xlabel('Count')
plt.show()

## 3. Sample High-Impact Conversations (Escalations vs Auto)
Notice how critical keywords like 'unauthorized', 'hacked', or legal threats signal high escalation needs.

In [ ]:
for intent in ['ACCOUNT_ACCESS', 'BILLING_CHARGE', 'DELIVERY_PROBLEM']:
    sample = df[df['intent'] == intent].head(2)
    print(f'=== {intent} ===')
    for _, row in sample.iterrows():
        print(f"Cust: {row['customer_text']}")
        print(f"Rep:  {row['brand_reply']}\n")